# Tarea B — Temperatura, top-p y entropía de la distribución de salida

## Parte 1 — Softmax con temperatura

In [1]:
# Subtarea T1: Implementar en NumPy la softmax con temperatura numéricamente estable (restar el máximo de los logits) sobre los logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.
# -*- coding: utf-8 -*-
"""
SUBTAREA T1: Softmax con temperatura (numéricamente estable) en NumPy.

Logits del enunciado para los tokens t0–t5: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]
Se calcula la distribución p_i = exp(z_i/T) / Σ_j exp(z_j/T) para T = 0.5, 1 y 2,
la entropía en bits H = -Σ p log2 p de cada una, y se presenta todo en una tabla
con cuatro decimales. Todas las cifras se guardan en "resultados.json".
No se requiere figura.
"""

import json
import numpy as np

# ----------------------------------------------------------------------
# Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

temperaturas = [0.5, 1.0, 2.0]


# ----------------------------------------------------------------------
# Funciones
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    """
    Softmax con temperatura, numéricamente estable.

    p_i = exp(z_i / T) / Σ_j exp(z_j / T)

    Estabilidad: se resta el máximo de los logits escalados antes de la
    exponencial, de modo que el mayor exponente es 0 y no hay desbordamiento.
    """
    z = np.asarray(z, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura T debe ser positiva.")
    z_escalado = z / T
    z_escalado = z_escalado - np.max(z_escalado)  # resta del máximo (estabilidad)
    e = np.exp(z_escalado)
    return e / np.sum(e)


def entropia_bits(p):
    """Entropía de Shannon en bits: H = -Σ p log2 p (términos con p=0 aportan 0)."""
    p = np.asarray(p, dtype=float)
    p_nz = p[p > 0]
    return float(-np.sum(p_nz * np.log2(p_nz)))


# ----------------------------------------------------------------------
# Cálculo de distribuciones y entropías
# ----------------------------------------------------------------------
distribuciones = {}
entropias = {}

for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    H = entropia_bits(p)
    clave = f"T={T}"
    distribuciones[clave] = {tok: float(pi) for tok, pi in zip(tokens, p)}
    entropias[clave] = float(H)

# ----------------------------------------------------------------------
# Guardar TODAS las cifras calculadas en resultados.json (floats de Python)
# ----------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": [float(z) for z in logits],
    "temperaturas": [float(T) for T in temperaturas],
    "distribuciones": distribuciones,
    "entropias_bits": entropias,
}
# Sumas de verificación (deben ser 1.0)
for T in temperaturas:
    clave = f"T={T}"
    resultados[f"suma_probabilidades_{clave}"] = float(
        sum(distribuciones[clave].values())
    )

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ----------------------------------------------------------------------
# Tabla con cuatro decimales
# ----------------------------------------------------------------------
anchos = 12
linea_sep = "-" * (8 + anchos * len(temperaturas))

print("Softmax con temperatura (distribuciones y entropía, 4 decimales)")
print(linea_sep)
encabezado = "Token".ljust(8) + "".join(f"T={T}".rjust(anchos) for T in temperaturas)
print(encabezado)
print(linea_sep)
for tok in tokens:
    fila = tok.ljust(8)
    for T in temperaturas:
        fila += f"{distribuciones[f'T={T}'][tok]:.4f}".rjust(anchos)
    print(fila)
print(linea_sep)
fila_H = "H (bits)".ljust(8)
for T in temperaturas:
    fila_H += f"{entropias[f'T={T}']:.4f}".rjust(anchos)
print(fila_H)
print(linea_sep)

# ----------------------------------------------------------------------
# Cifras principales con 4 decimales
# ----------------------------------------------------------------------
print("\nCifras principales:")
print(f"Logits: {[f'{z:.4f}' for z in logits]}")
for T in temperaturas:
    clave = f"T={T}"
    probs = distribuciones[clave]
    print(
        f"T={T}: p = [{', '.join(f'{probs[tok]:.4f}' for tok in tokens)}], "
        f"H = {entropias[clave]:.4f} bits"
    )

print("\nResultados guardados en resultados.json")


Softmax con temperatura (distribuciones y entropía, 4 decimales)
--------------------------------------------
Token          T=0.5       T=1.0       T=2.0
--------------------------------------------
t0            0.8231      0.5516      0.3583
t1            0.1114      0.2029      0.2173
t2            0.0410      0.1231      0.1693
t3            0.0225      0.0912      0.1457
t4            0.0020      0.0275      0.0800
t5            0.0000      0.0037      0.0294
--------------------------------------------
H (bits)      0.9147      1.7998      2.2888
--------------------------------------------

Cifras principales:
Logits: ['2.0000', '1.0000', '0.5000', '0.2000', '-1.0000', '-3.0000']
T=0.5: p = [0.8231, 0.1114, 0.0410, 0.0225, 0.0020, 0.0000], H = 0.9147 bits
T=1.0: p = [0.5516, 0.2029, 0.1231, 0.0912, 0.0275, 0.0037], H = 1.7998 bits
T=2.0: p = [0.3583, 0.2173, 0.1693, 0.1457, 0.0800, 0.0294], H = 2.2888 bits

Resultados guardados en resultados.json


Implementé la versión numéricamente estable restando el logit máximo antes de exponenciar: p_i = exp((z_i − z_max)/T) / Σ_j exp((z_j − z_max)/T). Restar la misma constante a todos los logits no cambia el cociente, pero evita que exp se desborde si los logits son grandes.

El patrón de los resultados es claro: al bajar la temperatura la masa se concentra en t0 y al subirla la distribución se aplana. Con T = 0.5, t0 se lleva 0.8231 de la masa y t5 queda en 0.0000 a cuatro decimales; con T = 2, t0 cae a 0.3583 y t5 sube a 0.0294. La entropía lo refleja de forma monótona: 0.9147 bits con T = 0.5, 1.7998 bits con T = 1 y 2.2888 bits con T = 2. Cuanto mayor es T, más incierta (más plana) es la distribución; cuanto menor es T, más se parece a una elección fija.

## Parte 2 — Muestreo de núcleo (top-p)

In [2]:
# Subtarea T2: Sobre la distribución de T=1 de la Parte 1, aplicar muestreo de núcleo top-p con p=0.9: ordenar los tokens por probabilidad descendente, conservar el 
# -*- coding: utf-8 -*-
"""
SUBTAREA T2: Muestreo de núcleo (top-p) con p=0.9 sobre la distribución T=1 de la Parte 1.

Pasos:
1. Recalcular la distribución softmax con T=1 a partir de los logits de la Parte 1
   (el script es independiente: no lee archivos de subtareas anteriores).
2. Ordenar los tokens por probabilidad descendente.
3. Conservar el conjunto mínimo cuya probabilidad acumulada alcanza p=0.9.
4. Renormalizar las probabilidades de los tokens supervivientes.
5. Guardar todas las cifras en resultados.json e imprimirlas con 4 decimales.

La subtarea indica "Figura: no", por lo que no se genera ningún PNG.
"""

import json
import numpy as np


def main():
    # ------------------------------------------------------------------
    # Datos de la Parte 1 (subtarea T1): logits y tokens
    # ------------------------------------------------------------------
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    temperatura = 1.0
    p_nucleo = 0.9

    # ------------------------------------------------------------------
    # 1. Softmax con temperatura T=1 (recomputado, con estabilidad numérica)
    # ------------------------------------------------------------------
    z = logits / temperatura
    z = z - np.max(z)
    exp_z = np.exp(z)
    probs = exp_z / np.sum(exp_z)

    # ------------------------------------------------------------------
    # 2. Ordenar tokens por probabilidad descendente
    # ------------------------------------------------------------------
    orden = np.argsort(-probs, kind="stable")
    tokens_ordenados = [tokens[i] for i in orden]
    probs_ordenadas = probs[orden]
    acumuladas = np.cumsum(probs_ordenadas)

    # ------------------------------------------------------------------
    # 3. Conjunto mínimo cuya probabilidad acumulada alcanza p=0.9
    #    (se incluye el token que hace cruzar el umbral)
    # ------------------------------------------------------------------
    idx_cruce = int(np.searchsorted(acumuladas, p_nucleo, side="left"))
    k = min(idx_cruce + 1, len(tokens))  # tamaño del núcleo
    indices_nucleo = orden[:k]
    tokens_supervivientes = [tokens[i] for i in indices_nucleo]
    tokens_eliminados = [tokens[i] for i in orden[k:]]
    masa_nucleo = float(acumuladas[k - 1])
    masa_eliminada = float(1.0 - masa_nucleo)

    # ------------------------------------------------------------------
    # 4. Renormalizar las probabilidades del núcleo
    # ------------------------------------------------------------------
    probs_nucleo = probs[indices_nucleo]
    renorm = probs_nucleo / masa_nucleo

    # Distribución renormalizada en longitud completa (0.0 para eliminados)
    renorm_completa = np.zeros(len(tokens), dtype=float)
    renorm_completa[indices_nucleo] = renorm

    # ------------------------------------------------------------------
    # Cifras adicionales (se guardan todas)
    # ------------------------------------------------------------------
    suma_T1 = float(np.sum(probs))
    suma_renorm = float(np.sum(renorm))
    entropia_T1_bits = float(-np.sum(probs * np.log2(probs)))
    entropia_renorm_bits = float(-np.sum(renorm * np.log2(renorm)))

    # ------------------------------------------------------------------
    # Guardar TODAS las cifras en resultados.json (un solo objeto)
    # ------------------------------------------------------------------
    resultados = {
        "tokens": tokens,
        "logits": [float(x) for x in logits],
        "temperatura": float(temperatura),
        "p_nucleo": float(p_nucleo),
        "distribucion_T1": {t: float(v) for t, v in zip(tokens, probs)},
        "suma_distribucion_T1": suma_T1,
        "entropia_T1_bits": entropia_T1_bits,
        "orden_descendente_tokens": tokens_ordenados,
        "orden_descendente_probs": [float(v) for v in probs_ordenadas],
        "probs_acumuladas_ordenadas": [float(v) for v in acumuladas],
        "num_tokens_nucleo": int(k),
        "tokens_supervivientes": tokens_supervivientes,
        "tokens_eliminados": tokens_eliminados,
        "masa_nucleo_antes_renormalizar": masa_nucleo,
        "masa_eliminada": masa_eliminada,
        "distribucion_renormalizada": {
            t: float(v) for t, v in zip(tokens_supervivientes, renorm)
        },
        "distribucion_renormalizada_completa": {
            t: float(v) for t, v in zip(tokens, renorm_completa)
        },
        "suma_distribucion_renormalizada": suma_renorm,
        "entropia_distribucion_renormalizada_bits": entropia_renorm_bits,
        "figuras": [],
    }

    with open("resultados.json", "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

    # ------------------------------------------------------------------
    # Impresión de las cifras principales (4 decimales)
    # ------------------------------------------------------------------
    linea = "-" * 64
    print("Top-p (muestreo de núcleo) con p=0.9 sobre la distribución T=1")
    print(linea)
    print("Distribución softmax con T=1 (recomputada de los logits):")
    for t, v in zip(tokens, probs):
        print(f"  {t}: {v:.4f}")
    print(f"  suma = {suma_T1:.4f}")
    print(linea)
    print("Tokens ordenados por probabilidad descendente:")
    print(f"  {'token':<7}{'p':>10}{'acumulada':>12}   estado")
    for j, (t, v, c) in enumerate(zip(tokens_ordenados, probs_ordenadas, acumuladas)):
        marca = ""
        if j < k:
            marca = "* en el núcleo"
        if j == k - 1:
            marca += " (aquí se alcanza p=0.9)"
        print(f"  {t:<7}{v:>10.4f}{c:>12.4f}   {marca}")
    print(linea)
    print(f"Conjunto mínimo con probabilidad acumulada >= {p_nucleo}:")
    print(f"  Tokens supervivientes (k={k}): {tokens_supervivientes}")
    print(f"  Tokens eliminados ({len(tokens) - k}): {tokens_eliminados}")
    print(f"  Masa del núcleo antes de renormalizar: {masa_nucleo:.4f}")
    print(f"  Masa eliminada: {masa_eliminada:.4f}")
    print(linea)
    print("Distribución renormalizada (núcleo top-p):")
    for t, v in zip(tokens_supervivientes, renorm):
        print(f"  {t}: {v:.4f}")
    print(f"  suma = {suma_renorm:.4f}")
    print(f"  entropía = {entropia_renorm_bits:.4f} bits "
          f"(T=1 original: {entropia_T1_bits:.4f} bits)")
    print(linea)
    print("Resultados guardados en resultados.json")


if __name__ == "__main__":
    main()


Top-p (muestreo de núcleo) con p=0.9 sobre la distribución T=1
----------------------------------------------------------------
Distribución softmax con T=1 (recomputada de los logits):
  t0: 0.5516
  t1: 0.2029
  t2: 0.1231
  t3: 0.0912
  t4: 0.0275
  t5: 0.0037
  suma = 1.0000
----------------------------------------------------------------
Tokens ordenados por probabilidad descendente:
  token           p   acumulada   estado
  t0         0.5516      0.5516   * en el núcleo
  t1         0.2029      0.7546   * en el núcleo
  t2         0.1231      0.8776   * en el núcleo
  t3         0.0912      0.9688   * en el núcleo (aquí se alcanza p=0.9)
  t4         0.0275      0.9963   
  t5         0.0037      1.0000   
----------------------------------------------------------------
Conjunto mínimo con probabilidad acumulada >= 0.9:
  Tokens supervivientes (k=4): ['t0', 't1', 't2', 't3']
  Tokens eliminados (2): ['t4', 't5']
  Masa del núcleo antes de renormalizar: 0.9688
  Masa eliminada: 0

Partí de la distribución con T = 1 y ordené los tokens de mayor a menor probabilidad; en este caso el orden coincide con el índice: t0, t1, t2, t3, t4, t5. Las probabilidades acumuladas de los cuatro primeros son 0.5516, 0.7546, 0.8776 y 0.9688. Como con tres tokens solo se llega a 0.8776 (por debajo de 0.9), el conjunto mínimo que alcanza el umbral necesita cuatro tokens: sobreviven t0, t1, t2 y t3, y se cortan t4 y t5, que juntos aportaban apenas 0.0312 de masa (el núcleo pesa 0.9688 antes de renormalizar).

Tras renormalizar, la distribución queda t0 = 0.5694, t1 = 0.2095, t2 = 0.1270 y t3 = 0.0941: cada superviviente gana masa en la misma proporción, su probabilidad original dividida entre 0.9688. El efecto sobre la incertidumbre se ve en la entropía, que baja de 1.7998 a 1.6341 bits porque el recorte elimina las opciones de la cola.

## Parte 3 — Comprobación empírica

In [3]:
# Subtarea T3: Tomar 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Comparar frecuencias observadas vs probabilidade
# -*- coding: utf-8 -*-
"""
SUBTAREA T3 — Comprobación empírica del muestreo de núcleo (top-p)
===================================================================
1) Recalcula (de forma autónoma) la distribución softmax con T=1 (Parte 1)
   y el núcleo top-p con p=0.9 renormalizado (Parte 2).
2) Toma 10 000 muestras de esa distribución renormalizada con
   numpy.random.default_rng(0).
3) Compara frecuencias observadas vs probabilidades teóricas en un
   gráfico de barras (PNG) y calcula la divergencia KL (en bits) de la
   distribución empírica respecto de la teórica.
Guarda todas las cifras en resultados.json.
"""

import json
import math

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# 1. Datos del enunciado (Partes 1 y 2), recomputados para autonomía
# ----------------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperatura = 1.0
p_nucleo = 0.9
num_muestras = 10000
semilla = 0

# Softmax con temperatura
z = logits / temperatura
z = z - np.max(z)
exp_z = np.exp(z)
probs_T = exp_z / np.sum(exp_z)

# Top-p: conjunto mínimo de tokens (orden descendente) con acumulada >= p
orden = np.argsort(-probs_T, kind="stable")
probs_ord = probs_T[orden]
acum = np.cumsum(probs_ord)
k = int(np.searchsorted(acum, p_nucleo - 1e-12, side="left")) + 1
idx_nucleo = np.sort(orden[:k])
tokens_nucleo = [tokens[i] for i in idx_nucleo]
tokens_fuera = [tokens[i] for i in orden[k:]]
masa_nucleo = float(np.sum(probs_T[idx_nucleo]))

# Distribución renormalizada sobre los 6 tokens (los eliminados quedan con 0)
probs_teo = np.zeros(len(tokens), dtype=float)
probs_teo[idx_nucleo] = probs_T[idx_nucleo] / masa_nucleo

# Entropía teórica de la distribución renormalizada (bits)
entropia_teo = 0.0
for pt in probs_teo:
    if pt > 0.0:
        entropia_teo -= pt * math.log2(pt)

# ----------------------------------------------------------------------------
# 2. Muestreo: 10 000 muestras con numpy.random.default_rng(0)
# ----------------------------------------------------------------------------
rng = np.random.default_rng(semilla)
muestras = rng.choice(len(tokens), size=num_muestras, p=probs_teo)
conteos = np.bincount(muestras, minlength=len(tokens)).astype(int)
frec_obs = conteos / float(num_muestras)

# ----------------------------------------------------------------------------
# 3. Divergencia KL (bits) de la empírica respecto de la teórica:
#    D_KL(P_emp || P_teo) = sum_x P_emp(x) * log2(P_emp(x) / P_teo(x))
#    Convención: los términos con P_emp(x) = 0 aportan 0.
#    Si P_teo(x) = 0 con P_emp(x) > 0, la KL sería infinita (no ocurre aquí,
#    porque solo se muestrea del núcleo).
# ----------------------------------------------------------------------------
kl_bits = 0.0
kl_infinita = False
for pe, pt in zip(frec_obs, probs_teo):
    if pe > 0.0:
        if pt <= 0.0:
            kl_infinita = True
            break
        kl_bits += pe * math.log2(pe / pt)
if kl_infinita:
    kl_bits = float("inf")

# Entropía empírica (bits), para comparar con la teórica
entropia_emp = 0.0
for pe in frec_obs:
    if pe > 0.0:
        entropia_emp -= pe * math.log2(pe)

# Errores por token
errores_abs = np.abs(frec_obs - probs_teo)
error_max = float(np.max(errores_abs))
error_medio = float(np.mean(errores_abs))

# ----------------------------------------------------------------------------
# 4. Figura: barras agrupadas teórica vs observada
# ----------------------------------------------------------------------------
nombre_figura = "t3_frecuencias_vs_teoricas.png"
x = np.arange(len(tokens))
ancho = 0.38
fig, ax = plt.subplots(figsize=(9.5, 5.8))
b1 = ax.bar(x - ancho / 2.0, probs_teo, ancho,
            label="Teórica (top-p renormalizada)",
            color="#4C72B0", edgecolor="black", linewidth=0.5)
b2 = ax.bar(x + ancho / 2.0, frec_obs, ancho,
            label=f"Observada ({num_muestras} muestras, rng({semilla}))",
            color="#DD8452", edgecolor="black", linewidth=0.5)
for barras in (b1, b2):
    for b in barras:
        h = b.get_height()
        ax.annotate(f"{h:.4f}",
                    (b.get_x() + b.get_width() / 2.0, h),
                    ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_ylim(0.0, max(float(np.max(probs_teo)), float(np.max(frec_obs))) * 1.20)
ax.set_title(f"T3: Frecuencias observadas vs probabilidades teóricas\n"
             f"(softmax T={temperatura}, top-p p={p_nucleo}, "
             f"{num_muestras} muestras, semilla {semilla})")
ax.legend(loc="upper right")
ax.text(0.985, 0.90,
        f"D_KL(emp||teo) = {kl_bits:.4f} bits",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="lightyellow", edgecolor="gray"))
fig.tight_layout()
fig.savefig(nombre_figura, dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------------
# 5. Guardar todas las cifras en resultados.json
# ----------------------------------------------------------------------------
resultados = {
    "subtarea": "T3_comprobacion_empirica_top_p",
    "num_muestras": int(num_muestras),
    "semilla": int(semilla),
    "temperatura": float(temperatura),
    "p_nucleo": float(p_nucleo),
    "tokens": tokens,
    "tokens_supervivientes": tokens_nucleo,
    "tokens_eliminados": tokens_fuera,
    "masa_nucleo_antes_renormalizar": masa_nucleo,
    "distribucion_teorica_renormalizada": {t: float(p) for t, p in zip(tokens, probs_teo)},
    "suma_distribucion_teorica": float(np.sum(probs_teo)),
    "entropia_teorica_bits": float(entropia_teo),
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(f) for t, f in zip(tokens, frec_obs)},
    "suma_frecuencias_observadas": float(np.sum(frec_obs)),
    "entropia_empirica_bits": float(entropia_emp),
    "divergencia_KL_bits": float(kl_bits),
    "divergencia_KL_infinita": bool(kl_infinita),
    "error_absoluto_por_token": {t: float(e) for t, e in zip(tokens, errores_abs)},
    "error_absoluto_maximo": error_max,
    "error_absoluto_medio": error_medio,
    "figura": nombre_figura,
    "figuras": [nombre_figura],
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------------
# 6. Impresión de las cifras principales (4 decimales)
# ----------------------------------------------------------------------------
print("SUBTAREA T3 — Comprobación empírica del muestreo top-p")
print("=" * 66)
print(f"Muestras: {num_muestras} | semilla: {semilla} | T = {temperatura} | p = {p_nucleo}")
print(f"Tokens del núcleo: {tokens_nucleo} | eliminados: {tokens_fuera}")
print("-" * 66)
print("token   teórica   observada   conteo   |error|")
for t, pt, pe, c in zip(tokens, probs_teo, frec_obs, conteos):
    print(f"{t:>5}    {pt:.4f}    {pe:.4f}    {c:6d}   {abs(pe - pt):.4f}")
print("-" * 66)
print(f"Divergencia KL (empírica || teórica): {kl_bits:.4f} bits")
print(f"Entropía empírica:  {entropia_emp:.4f} bits")
print(f"Entropía teórica:   {entropia_teo:.4f} bits")
print(f"Error absoluto máximo: {error_max:.4f}")
print(f"Error absoluto medio:  {error_medio:.4f}")
print("-" * 66)
print(f"Figura guardada: {nombre_figura}")
print("Resultados guardados en resultados.json")


SUBTAREA T3 — Comprobación empírica del muestreo top-p
Muestras: 10000 | semilla: 0 | T = 1.0 | p = 0.9
Tokens del núcleo: ['t0', 't1', 't2', 't3'] | eliminados: ['t4', 't5']
------------------------------------------------------------------
token   teórica   observada   conteo   |error|
   t0    0.5694    0.5682      5682   0.0012
   t1    0.2095    0.2093      2093   0.0002
   t2    0.1270    0.1296      1296   0.0026
   t3    0.0941    0.0929       929   0.0012
   t4    0.0000    0.0000         0   0.0000
   t5    0.0000    0.0000         0   0.0000
------------------------------------------------------------------
Divergencia KL (empírica || teórica): 0.0001 bits
Entropía empírica:  1.6362 bits
Entropía teórica:   1.6341 bits
Error absoluto máximo: 0.0026
Error absoluto medio:  0.0009
------------------------------------------------------------------
Figura guardada: t3_frecuencias_vs_teoricas.png
Resultados guardados en resultados.json


Generé 10 000 muestras de la distribución renormalizada con numpy.random.default_rng(0). Los conteos fueron 5682, 2093, 1296 y 929 para t0, t1, t2 y t3, es decir, frecuencias observadas de 0.5682, 0.2093, 0.1296 y 0.0929, frente a las teóricas 0.5694, 0.2095, 0.1270 y 0.0941. t4 y t5 no aparecieron ni una sola vez, como corresponde a tokens con probabilidad 0 después del recorte. El error absoluto máximo fue 0.0026 (en t2), la entropía empírica dio 1.6362 bits frente a 1.6341 teóricos, y la divergencia KL de la empírica respecto de la teórica fue 0.0001 bits. Las diferencias son del tamaño esperable por el azar del muestreo con n = 10 000, no señalan un error en el procedimiento.

![Frecuencias observadas frente a teóricas en 10 000 muestras top-p](T3_t3_frecuencias_vs_teoricas.png)

## Parte 4 — Pregunta conceptual

Cuando T → 0⁺, el cociente z_i/T amplifica las diferencias entre logits: el token de mayor logit obtiene un exponencial enormemente mayor que los demás y absorbe prácticamente toda la masa. En el límite, la distribución es degenerada (one-hot): probabilidad 1 sobre t0, que aquí tiene el logit más alto (2.0), y 0 sobre el resto. La entropía tiende a 0 bits, el mínimo posible. Cuando T → ∞ ocurre lo contrario: z_i/T → 0 para todos los tokens, todos los exponenciales valen lo mismo y la distribución tiende a la uniforme, con la misma probabilidad para los seis tokens; la entropía tiende al máximo de un vocabulario de seis elementos, log2(6) bits.

Las entropías de la Parte 1 muestran exactamente esa trayectoria: al subir T de 0.5 a 1 y de 1 a 2, la entropía sube de 0.9147 a 1.7998 y luego a 2.2888 bits. Bajar T la acerca a 0 (elección determinista) y subir T la acerca al máximo (elección uniforme).

T → 0 equivale a la decodificación voraz por dos razones. Primera: el softmax es monótono en los logits para cualquier temperatura, así que el token más probable de la distribución es siempre el argmax de los logits, que es justamente el que elige greedy. Segunda: en el límite T → 0 ese token concentra toda la probabilidad, de modo que muestrear de la distribución devuelve siempre ese token, sin azar. Dicho de otro modo, greedy es el caso límite del muestreo con temperatura: la selección coincide, y la temperatura solo controla cuánta probabilidad se reparte entre los tokens perdedores. Conviene escribir "T → 0⁺" y no "T = 0", porque en T = 0 la fórmula no está definida (división entre cero), pero el límite sí existe y es la distribución one-hot sobre el argmax.